In [1]:
import pandas as pd
import numpy as np
import openpyxl

import libs.append_path
from add_python_libraries import *
import pd_filter_fcns

from cyntec import Inductor_pdis, lparams,create_ind_family_df, ind_pdis_obj, l_set, cyntec_filename
from circuit_4state import circuit_params as circuit_params_4state
from circuit_2state import circuit_params as circuit_params_2state

In [2]:
input_params = {'vin' : 28,
                'vout': 9,
                'pin' : 165, 
                'eff' : .97, 
                'fs'  :300e3,
                'ton_mult':1,
                'tambient':45,
                'lout':{'family'   : 'cmll063t', #'cmlb104t',
                        'value(uH)':0.4,
                        'config'   :'single'},
                'lvl_config':'3 level',
               }
input_params['iout']=input_params['pin']*input_params['eff']/input_params['vout']

### whiteboard

In [8]:
ckt = circuit_params_4state(input_params)

In [ ]:
ckt

In [17]:
ind = 0.4

In [18]:
ipp       = abs(ckt['deltaV']*ckt['t_for_deltaV']/ind/1e-6) 

In [19]:
ipp

13.392857142857142

### code

In [30]:
class Inductor:
    def __init__(self,ip,lout,vout):
        self.ip = ip
        self.ip['vout']=vout
        self.ip['iout']=self.ip['pin']*self.ip['eff']/vout
        self.lout = lout
        self.ckt = circuit_params_4state(self.ip)
        self.idc = round(self.ckt['Idc'],3)
        self.ipp = round(abs(self.ckt['deltaV']*self.ckt['t_for_deltaV']/self.lout/1e-6),3) 
        self.summary = {'idc':self.idc,
                        'ipp':self.ipp}

In [37]:
ind_obj = Inductor(input_params,1,9)

In [38]:
ind_obj.idc

17.78333333333333

In [31]:
lout_list = [0.4, 0.47, 0.68, 1.0]
vout_list = [9,12,18]
ind_obj_dict = {lout:{vout:Inductor(input_params,lout,vout).summary
                for vout in vout_list}
                for lout in lout_list}

In [32]:
ind_obj_dict

{0.4: {9: {'idc': 17.783, 'ipp': 13.393},
  12: {'idc': 13.337, 'ipp': 7.143},
  18: {'idc': 8.892, 'ipp': 11.905}},
 0.47: {9: {'idc': 17.783, 'ipp': 11.398},
  12: {'idc': 13.337, 'ipp': 6.079},
  18: {'idc': 8.892, 'ipp': 10.132}},
 0.68: {9: {'idc': 17.783, 'ipp': 7.878},
  12: {'idc': 13.337, 'ipp': 4.202},
  18: {'idc': 8.892, 'ipp': 7.003}},
 1.0: {9: {'idc': 17.783, 'ipp': 5.357},
  12: {'idc': 13.337, 'ipp': 2.857},
  18: {'idc': 8.892, 'ipp': 4.762}}}

In [33]:
df = pd.DataFrame.from_dict(ind_obj_dict)

In [34]:
df

,0.40,0.47,0.68,1.00
9,"{'idc': 17.783, 'ipp': 13.393}","{'idc': 17.783, 'ipp': 11.398}","{'idc': 17.783, 'ipp': 7.878}","{'idc': 17.783, 'ipp': 5.357}"
12,"{'idc': 13.337, 'ipp': 7.143}","{'idc': 13.337, 'ipp': 6.079}","{'idc': 13.337, 'ipp': 4.202}","{'idc': 13.337, 'ipp': 2.857}"
18,"{'idc': 8.892, 'ipp': 11.905}","{'idc': 8.892, 'ipp': 10.132}","{'idc': 8.892, 'ipp': 7.003}","{'idc': 8.892, 'ipp': 4.762}"


In [38]:
df.iloc[:,:]

,0.40,0.47,0.68,1.00
9,"{'idc': 17.783, 'ipp': 13.393}","{'idc': 17.783, 'ipp': 11.398}","{'idc': 17.783, 'ipp': 7.878}","{'idc': 17.783, 'ipp': 5.357}"
12,"{'idc': 13.337, 'ipp': 7.143}","{'idc': 13.337, 'ipp': 6.079}","{'idc': 13.337, 'ipp': 4.202}","{'idc': 13.337, 'ipp': 2.857}"
18,"{'idc': 8.892, 'ipp': 11.905}","{'idc': 8.892, 'ipp': 10.132}","{'idc': 8.892, 'ipp': 7.003}","{'idc': 8.892, 'ipp': 4.762}"


In [41]:
idc_dict = {lout:{vout:Inductor(input_params,lout,vout).idc
                for vout in vout_list}
                for lout in lout_list}
ipp_dict = {lout:{vout:Inductor(input_params,lout,vout).ipp
                for vout in vout_list}
                for lout in lout_list}

In [43]:
df_idc = pd.DataFrame.from_dict(idc_dict)
df_ipp = pd.DataFrame.from_dict(ipp_dict)

In [44]:
df_idc

,0.40,0.47,0.68,1.00
9,17.783,17.783,17.783,17.783
12,13.337,13.337,13.337,13.337
18,8.892,8.892,8.892,8.892


In [45]:
df_ipp

,0.40,0.47,0.68,1.00
9,13.393,11.398,7.878,5.357
12,7.143,6.079,4.202,2.857
18,11.905,10.132,7.003,4.762
